# 3 · Model training: Inception V3 with transfer learning

All code is written out in this notebook, so every step can be read and changed here.

**Plan**
1. Settings
2. Dataset: read images, augment, normalise
3. Model: pre-trained Inception V3 with a new output layer
4. Loss function, optimiser, learning-rate schedule
5. Training loop
6. Training curves

**Transfer learning:** Inception V3 already learned edges, textures and shapes from 1.2 million ImageNet photos.
We keep those layers **frozen** and retrain only the last 2 blocks plus a new output layer (benign / malignant).
This works with little data and is fast enough for a CPU (~2–3 min per epoch).

## 1. Imports and settings

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm.auto import tqdm

In [ ]:
# ---- folders (this notebook lives in notebooks/, the project root is one level up)
ROOT = Path("..").resolve()
ENHANCED_DIR = ROOT / "data" / "enhanced"     # ODTWCHE-enhanced images (notebook 02)
MASKS_DIR = ROOT / "data" / "masks"           # tumour outlines drawn by radiologists
SPLITS_DIR = ROOT / "data" / "splits"         # train.csv / val.csv / test.csv (notebook 01)
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ["benign", "malignant"]
IMAGE_SIZE = 299            # Inception V3 expects 299 x 299 images
BATCH_SIZE = 16

# ---- training settings (paper Table 3; fewer epochs because we train on a CPU)
EPOCHS = 10               # paper: 30
LEARNING_RATE = 0.001
MOMENTUM = 0.9            # paper: SGD with momentum 0.9
WEIGHT_DECAY = 0.0001     # paper: L2 regularisation 0.0001
LR_DROP_EVERY = 4         # paper: learning rate x 0.1 every 10 epochs (of 30)
SEED = 42

torch.manual_seed(SEED)   # same random numbers every run -> reproducible results
np.random.seed(SEED)

## 2. Dataset

A PyTorch `Dataset` answers two questions: *how many images are there?* (`__len__`) and *give me image number i* (`__getitem__`).
For each image it:
1. loads the enhanced PNG and resizes it to 299×299
2. (training only) applies a random rotation / flip, so the model sees a slightly different image every epoch
3. copies the grey channel 3× because the network was trained on colour photos
4. normalises with the ImageNet mean and standard deviation

In [ ]:
# ImageNet mean / std: the pre-trained network was trained on images normalised like this
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class MRIDataset(Dataset):
    """Reads the images listed in train.csv / val.csv / test.csv."""

    def __init__(self, split, augment=False):
        self.df = pd.read_csv(SPLITS_DIR / f"{split}.csv")
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def load_image(self, i):
        img = cv2.imread(str(ENHANCED_DIR / self.df.file[i]), cv2.IMREAD_GRAYSCALE)
        return cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))

    def __getitem__(self, i):
        img = self.load_image(i)

        if self.augment:                                  # data augmentation (paper Sec. 3.1)
            img = np.rot90(img, k=np.random.randint(4))   # rotate 0 / 90 / 180 / 270 degrees
            if np.random.rand() < 0.5:
                img = np.fliplr(img)                      # horizontal flip
            if np.random.rand() < 0.5:
                img = np.flipud(img)                      # vertical flip

        x = np.stack([img, img, img], axis=-1)            # grey -> 3 channels (network expects RGB)
        x = x.astype(np.float32) / 255.0                  # 0..255 -> 0..1
        x = (x - MEAN) / STD                              # ImageNet normalisation
        x = torch.from_numpy(x.transpose(2, 0, 1).copy()) # (H, W, C) -> (C, H, W)
        y = int(self.df.label[i])                         # 0 = benign, 1 = malignant
        return x, y

In [ ]:
train_ds = MRIDataset("train", augment=True)
val_ds = MRIDataset("val")
print("train images:", len(train_ds), "  val images:", len(val_ds))
print(train_ds.df["class"].value_counts().to_dict())

What the network sees: the **same** training image with 6 different random augmentations:

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax in axes:
    x, y = train_ds[0]
    ax.imshow(x[0], cmap="gray")
    ax.set_title(CLASS_NAMES[y])
    ax.axis("off")

A `DataLoader` groups images into **batches** of 16 and shuffles the training set every epoch.

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

x, y = next(iter(train_loader))
print("one batch:", tuple(x.shape), "  labels:", y.tolist())

## 3. Model

Steps:
1. load Inception V3 with ImageNet weights
2. freeze all layers
3. replace the output layer (1000 ImageNet classes → 2 classes) and add dropout 0.2
4. un-freeze the last two Inception blocks (`Mixed_7b`, `Mixed_7c`)

Inception V3 also has a small **auxiliary classifier** in the middle of the network. It is used only during training
and helps gradients reach the earlier layers.

In [ ]:
def build_model(pretrained=True):
    # 1. load Inception V3 (with ImageNet weights when pretrained=True)
    model = models.inception_v3(weights="DEFAULT" if pretrained else None,
                                aux_logits=True, init_weights=not pretrained)

    # 2. freeze every layer
    for p in model.parameters():
        p.requires_grad = False

    # 3. replace the last layer: 1000 ImageNet classes -> 2 classes (benign / malignant)
    model.fc = nn.Sequential(nn.Dropout(0.2),           # dropout 0.2 (paper)
                             nn.Linear(2048, 2))
    model.AuxLogits.fc = nn.Linear(768, 2)              # the small auxiliary classifier too

    # 4. un-freeze the last two Inception blocks so they can adapt to MRI images
    for block in [model.Mixed_7b, model.Mixed_7c, model.AuxLogits]:
        for p in block.parameters():
            p.requires_grad = True
    return model

In [ ]:
model = build_model()

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters: {n_total:,} total, {n_train:,} trainable ({n_train / n_total:.0%})")

## 4. Loss function, optimiser, learning-rate schedule

* **Cross-entropy loss**: large when the model is confident and wrong, small when it is right
* **SGD with momentum 0.9**: the paper's optimiser. `weight_decay` = L2 regularisation (keeps weights small, reduces overfitting)
* **StepLR**: multiply the learning rate by 0.1 every few epochs, so big steps at the start and fine-tuning at the end

In [ ]:
loss_fn = nn.CrossEntropyLoss()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(trainable_params, lr=LEARNING_RATE,
                            momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_DROP_EVERY, gamma=0.1)

## 5. Training loop

**One training epoch** = show every training image once. For each batch:
1. forward pass → predictions
2. compute the loss (main output + 0.4 × auxiliary output)
3. `loss.backward()` → gradients (how much each weight contributed to the error)
4. `optimizer.step()` → update the weights a little

**Validation** = same forward pass on the validation images, but *without* updating weights
(`model.eval()` + `torch.no_grad()`). This shows how well the model works on patients it has not trained on.

Note: frozen BatchNorm layers are put back in eval mode so they keep their ImageNet statistics.

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    for m in model.modules():                        # frozen BatchNorm layers keep ImageNet statistics
        if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
            m.eval()

    total_loss, correct, count = 0.0, 0, 0
    for x, y in tqdm(loader, desc="train", leave=False):
        out = model(x)                               # in training mode: out.logits and out.aux_logits
        loss = loss_fn(out.logits, y) + 0.4 * loss_fn(out.aux_logits, y)

        optimizer.zero_grad()                        # clear old gradients
        loss.backward()                              # compute new gradients
        optimizer.step()                             # update weights

        total_loss += loss.item() * len(y)
        correct += (out.logits.argmax(dim=1) == y).sum().item()
        count += len(y)
    return total_loss / count, correct / count


def validate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():                            # no gradients needed -> faster
        for x, y in tqdm(loader, desc="val", leave=False):
            logits = model(x)                        # in eval mode: only the main output
            loss = loss_fn(logits, y)
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            count += len(y)
    return total_loss / count, correct / count

Run the training. After each epoch, the model is saved **if its validation accuracy is the best so far**.

In [ ]:
history = []
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)
    lr = optimizer.param_groups[0]["lr"]
    scheduler.step()                                 # maybe lower the learning rate

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                    "val_loss": val_loss, "val_acc": val_acc, "lr": lr})

    saved = ""
    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODELS_DIR / "inception_v3.pt")
        saved = "  <- best, saved"

    print(f"epoch {epoch:2d} | lr {lr:.0e} | train loss {train_loss:.3f} acc {train_acc:.3f} | "
          f"val loss {val_loss:.3f} acc {val_acc:.3f}{saved}")

history = pd.DataFrame(history)
history.to_csv(RESULTS_DIR / "history.csv", index=False)
print("best validation accuracy:", round(best_val_acc, 3))

## 6. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.epoch, history.train_loss, "o-", label="train")
axes[0].plot(history.epoch, history.val_loss, "o-", label="validation")
axes[0].set_title("loss")
axes[1].plot(history.epoch, history.train_acc, "o-", label="train")
axes[1].plot(history.epoch, history.val_acc, "o-", label="validation")
axes[1].set_title("accuracy")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=100)

**How to read the curves**
* both losses go down → the model is learning
* train accuracy keeps rising while validation accuracy stalls or drops → **overfitting** (the model memorises training images)
* validation is sometimes higher than train here because dropout and augmentation are only active during training

The best model is saved in `models/inception_v3.pt` → continue with notebook 04.